# 03 — Prepare the first batch
Run notebook 01 first. This notebook uses the exact preparation functions shared with the pipeline. It checks patient separation and fits preprocessing on training data only.

In [1]:
import sys
from pathlib import Path
project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from io import BytesIO


## Functions: settings
This is a copy; edit the source in 05_pipeline_walkthrough.ipynb. Run the cells in order. After saving edits in the source notebooks, run the synchronization command described in the README.

In [2]:
import os
from urllib.parse import quote_plus


In [3]:
def s3_client():
    import boto3
    return boto3.client('s3', endpoint_url=os.getenv('S3_ENDPOINT_URL','http://minio.mlops-project.svc.cluster.local:9000'), region_name='us-east-1', aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'], aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'])


## Functions: preprocessing
This is the editable source. Run the cells in order. After saving edits in the source notebooks, run the synchronization command described in the README.

In [4]:
import hashlib
import numpy as np
import pandas as pd
NUMERIC_FEATURES = (
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses',
)
SPLIT_STRATEGY = 'patient_nbr_sha256_70_15_15'


In [5]:
def split_for_patient(patient_id):
    value = int(hashlib.sha256(str(int(patient_id)).encode()).hexdigest()[:12], 16) % 10000
    return 'train' if value < 7000 else ('validation' if value < 8500 else 'test')


In [6]:
def normalize_features(features):
    """Keep categorical columns categorical, even when an entire batch is missing."""
    result = features.copy()
    result = result.mask(result.eq('?'))
    for column in result:
        if column in NUMERIC_FEATURES:
            result[column] = pd.to_numeric(result[column], errors='coerce').astype('float64')
        else:
            result[column] = result[column].map(
                lambda value: str(value) if pd.notna(value) else np.nan
            ).astype(object)
    return result


In [7]:
def prepare_records(df):
    required = {'encounter_id', 'patient_nbr', 'readmitted'}
    if not required.issubset(df.columns):
        raise ValueError(f'Missing columns: {required - set(df.columns)}')
    df = df.copy().reset_index(drop=True)
    df = df.mask(df.eq('?'))
    if df[['encounter_id', 'patient_nbr', 'readmitted']].isna().any().any():
        raise ValueError('Encounter ID, patient ID and readmission label are required')
    unexpected = set(df['readmitted'].unique()) - {'NO', '>30', '<30'}
    if unexpected:
        raise ValueError(f'Unexpected readmission labels: {unexpected}')
    df['target'] = df['readmitted'].eq('<30').astype('int8')
    df['split'] = df['patient_nbr'].map(split_for_patient)
    excluded = {'encounter_id', 'patient_nbr', 'readmitted', 'target', 'split', 'discharge_disposition_id'}
    features = normalize_features(df.drop(columns=[c for c in excluded if c in df.columns]))
    return df[['encounter_id', 'patient_nbr', 'target', 'split']], features


## Functions: training
This is a copy; edit the source in 04_modeling.ipynb. Run the cells in order. After saving edits in the source notebooks, run the synchronization command described in the README.

In [8]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix


In [9]:
def make_pipeline(frame,kind='logistic'):
    numbers=frame.select_dtypes(include='number').columns.tolist()
    categories=[c for c in frame.columns if c not in numbers]
    steps=[]
    if numbers: steps.append(('num',Pipeline([('fill',SimpleImputer(strategy='median',keep_empty_features=True)),('scale',StandardScaler())]),numbers))
    if categories: steps.append(('cat',Pipeline([('fill',SimpleImputer(strategy='constant',fill_value='missing',keep_empty_features=True)),('encode',OneHotEncoder(handle_unknown='ignore'))]),categories))
    prep=ColumnTransformer(steps)
    if kind not in ('logistic', 'forest'): raise ValueError(f'Unknown model: {kind}')
    model= LogisticRegression(max_iter=500,class_weight='balanced',random_state=42) if kind=='logistic' else RandomForestClassifier(n_estimators=100,min_samples_leaf=5,random_state=42,n_jobs=2,class_weight='balanced_subsample')
    return Pipeline([('preprocess',prep),('model',model)])


In [10]:
def score_predictions(y, predicted, probabilities):
    """Report counts and classification scores; undefined AUCs remain null."""
    y = np.asarray(y, dtype=int)
    predicted = np.asarray(predicted, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if len(y) == 0:
        return None
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    result = {
        'n': int(len(y)), 'positive': int((y == 1).sum()), 'negative': int((y == 0).sum()),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'accuracy': float(accuracy_score(y, predicted)),
        'precision': float(precision_score(y, predicted, zero_division=0)),
        'recall': float(recall_score(y, predicted, zero_division=0)),
        'f1': float(f1_score(y, predicted, zero_division=0)),
        'roc_auc': None, 'pr_auc': None,
    }
    if len(np.unique(y)) == 2:
        result['roc_auc'] = float(roc_auc_score(y, probabilities))
        # This is average precision, not trapezoidal PR-curve area.
        result['pr_auc'] = float(average_precision_score(y, probabilities))
    return result


def metrics(model, X, y):
    if len(y) == 0:
        return None
    positive_column = list(model.classes_).index(1)
    return score_predictions(y, model.predict(X), model.predict_proba(X)[:, positive_column])


In [11]:
body = s3_client().get_object(Bucket='diabetes-source', Key='batches/batch_0001.csv')['Body'].read()
data = pd.read_csv(BytesIO(body), dtype=str, keep_default_na=False)
data = data.sort_values('encounter_id', key=lambda col: pd.to_numeric(col), kind='stable').reset_index(drop=True)
meta, X = prepare_records(data)
train = meta['split'].eq('train')
valid = meta['split'].eq('validation')
test = meta['split'].eq('test')
print('Split strategy:', SPLIT_STRATEGY)
display(pd.crosstab(meta['split'], meta['target'], margins=True))

Split strategy: patient_nbr_sha256_70_15_15


target,0,1,All
split,,,
test,2115,274,2389
train,9304,1146,10450
validation,1928,233,2161
All,13347,1653,15000


In [12]:
patient_splits = meta.groupby('patient_nbr')['split'].nunique()
assert patient_splits.max() == 1, 'A patient appears in more than one split'
assert meta.loc[train, 'target'].nunique() == 2
print('No patients overlap between splits.')
print('Test rows reserved for a later predefined evaluation:', int(test.sum()))
print('Feature columns:', len(X.columns))
display(X.dtypes.astype(str).value_counts().rename_axis('dtype').to_frame('columns'))
display((X.isna().mean().sort_values(ascending=False).head(10) * 100).rename('missing_pct').to_frame())

No patients overlap between splits.
Test rows reserved for a later predefined evaluation: 2389
Feature columns: 46


,columns
dtype,
object,38
float64,8


,missing_pct
payer_code,100.000000
weight,96.926667
medical_specialty,38.960000
diag_3,2.860000
race,1.753333
diag_2,0.586667
diag_1,0.046667
admission_source_id,0.000000
time_in_hospital,0.000000
num_lab_procedures,0.000000


## Fit transformations using training data only
Continuous numeric values use median imputation and scaling. Categorical features, including admission codes, use a missing category and one-hot encoding. Unseen validation categories are handled without refitting.

In [13]:
transformer = make_pipeline(X.loc[train], 'logistic').named_steps['preprocess']
train_matrix = transformer.fit_transform(X.loc[train])
validation_matrix = transformer.transform(X.loc[valid])
print('Training matrix:', train_matrix.shape)
print('Validation matrix:', validation_matrix.shape)
print('Sample transformed feature names:')
print(transformer.get_feature_names_out()[:15])

Training matrix: (10450, 1512)
Validation matrix: (2161, 1512)
Sample transformed feature names:
['num__time_in_hospital' 'num__num_lab_procedures' 'num__num_procedures'
 'num__num_medications' 'num__number_outpatient' 'num__number_emergency'
 'num__number_inpatient' 'num__number_diagnoses'
 'cat__race_AfricanAmerican' 'cat__race_Asian' 'cat__race_Caucasian'
 'cat__race_Hispanic' 'cat__race_Other' 'cat__race_missing'
 'cat__gender_Female']


The fitted transformer above is only a demonstration. Each candidate model in notebook 04 and the pipeline fits its own preprocessing inside a scikit-learn Pipeline, using its training rows only.